# analysis of the similarity between train and test data in AM-I, AM-II

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from multiprocessing import Pool, cpu_count

# 相似度阈值列表（从高到低排序） - 修改为10个阈值
THRESHOLDS = [0.90, 0.80, 0.70, 0.60, 0.50, 0.4, 0.3, 0.2, 0.1, 0]
PERCENTILES = [50, 80, 95]  # 要输出的分位数
MAX_CORES = 26  # 并行计算核数

# 可自定义输出目录（默认 ./similarity）
OUTPUT_DIR = input("请输入输出目录路径（默认 ./3-similarity）: ").strip() or "./3-similarity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 定义train_test_split目录路径
DATA_DIR = './1-train_test_split/'

# 将 SMILES 转换为 Morgan 指纹
def smiles_to_fp(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)

# 计算单个 test 分子的最大相似度
def compute_max_sim(args):
    test_fp, train_fps = args
    if test_fp is None:
        return 0.0
    sims = DataStructs.BulkTanimotoSimilarity(test_fp, train_fps)
    return max(sims) if sims else 0.0

# 主函数
for file in os.listdir(DATA_DIR):
    if file.endswith('_test.csv'):
        dataset_name = os.path.splitext(file)[0]
        prefix = file.replace('_test.csv', '')
        train_file = os.path.join(DATA_DIR, prefix + '_train.csv')
        test_file = os.path.join(DATA_DIR, file)  # 完整的test文件路径

        if not os.path.exists(train_file):
            print(f"❌ 未找到对应的 train 文件: {train_file}")
            continue

        print(f"处理: {file} vs {train_file}")

        # 读取数据 - 使用完整路径
        test_df = pd.read_csv(test_file)
        train_df = pd.read_csv(train_file)

        # 转 fingerprint
        train_fps = [smiles_to_fp(s) for s in train_df['SMILES'].dropna()]
        train_fps = [fp for fp in train_fps if fp is not None]
        test_fps = [smiles_to_fp(s) for s in test_df['SMILES'].dropna()]

        # 并行计算最大相似度
        with Pool(min(MAX_CORES, cpu_count())) as pool:
            max_sims = pool.map(compute_max_sim, [(fp, train_fps) for fp in test_fps])

        # 添加到 DataFrame
        test_df["Max_Similarity"] = max_sims

        # 建立结果字典
        results = {thr: [] for thr in THRESHOLDS}
        for idx, sim in enumerate(max_sims):
            for thr in THRESHOLDS:
                if sim >= thr:
                    results[thr].append(idx)

        # 保存结果
        total_test = len(test_df)
        stats = []
        for thr in THRESHOLDS:
            indices = results[thr]
            if indices:
                out_df = test_df.iloc[indices]
                # 对于阈值0，使用特殊命名
                if thr == 0:
                    out_name = os.path.join(OUTPUT_DIR, f"{dataset_name}_sim0_all.csv")
                else:
                    out_name = os.path.join(OUTPUT_DIR, f"{dataset_name}_sim{int(thr*100) if thr >= 0.1 else int(thr*10)}.csv")
                out_df.to_csv(out_name, index=False)
                count = len(out_df)
                stats.append((thr, count, count / total_test))
                print(f"阈值 {thr:.2f}: 样本数={count}, 占比={count/total_test:.2%}, 已保存 {out_name}")

        # 分位数
        percentiles_values = np.percentile(max_sims, PERCENTILES)
        percentile_stats = [(f"P{p}", val) for p, val in zip(PERCENTILES, percentiles_values)]

        # 保存统计结果
        stats_df = pd.DataFrame(stats, columns=["Threshold", "Count", "Proportion"])
        percentiles_df = pd.DataFrame(percentile_stats, columns=["Percentile", "Similarity"])
        stats_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_similarity_stats.xlsx")
        with pd.ExcelWriter(stats_path) as writer:
            stats_df.to_excel(writer, sheet_name="Threshold_Stats", index=False)
            percentiles_df.to_excel(writer, sheet_name="Percentiles", index=False)
        print(f"📑 已保存统计结果和分位数: {stats_path}")

        # 绘图直方图
        plt.figure(figsize=(10, 6))
        plt.hist(max_sims, bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel("Maximum Similarity to Train Set")
        plt.ylabel("Frequency")
        plt.title(f"{dataset_name} - Max Similarity Distribution")
        plt.tight_layout()
        hist_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_max_similarity_hist.png")
        plt.savefig(hist_path, dpi=300)
        plt.close()
        print(f"📊 已保存最大相似度分布直方图: {hist_path}")

        # 绘制CDF并标注分位点
        sorted_sims = np.sort(max_sims)
        cdf = np.arange(1, len(sorted_sims)+1) / len(sorted_sims)

        plt.figure(figsize=(10, 6))
        plt.plot(sorted_sims, cdf, marker='.', linestyle='-', label='CDF')
        for p, val in zip(PERCENTILES, percentiles_values):
            plt.axvline(x=val, linestyle='--', alpha=0.7, label=f"P{p}={val:.2f}")
            plt.scatter([val], [p/100], color='red')
            plt.text(val, p/100, f" P{p}", fontsize=9, verticalalignment='bottom')
        plt.xlabel("Maximum Similarity to Train Set")
        plt.ylabel("Cumulative Proportion")
        plt.title(f"{dataset_name} - Max Similarity CDF")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        cdf_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_max_similarity_cdf.png")
        plt.savefig(cdf_path, dpi=600)
        plt.close()
        print(f"📈 已保存最大相似度累积分布曲线: {cdf_path}")

print("✅ 所有文件处理完成!")

处理: AM-II-filtered_with_labels_k4_test.csv vs ./1-train_test_split/AM-II-filtered_with_labels_k4_train.csv


[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerator
[23:22:07] DEPRECATION WARNING: please use MorganGenerat

阈值 0.90: 样本数=3, 占比=1.61%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim90.csv
阈值 0.80: 样本数=12, 占比=6.45%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim80.csv
阈值 0.70: 样本数=50, 占比=26.88%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim70.csv
阈值 0.60: 样本数=113, 占比=60.75%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim60.csv
阈值 0.50: 样本数=168, 占比=90.32%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim50.csv
阈值 0.40: 样本数=183, 占比=98.39%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim40.csv
阈值 0.30: 样本数=186, 占比=100.00%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim30.csv
阈值 0.20: 样本数=186, 占比=100.00%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim20.csv
阈值 0.10: 样本数=186, 占比=100.00%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim10.csv
阈值 0.00: 样本数=186, 占比=100.00%, 已保存 ./3-similarity/AM-II-filtered_with_labels_k4_test_sim0_all.csv
📑 已保存统计结果和分位数: ./3-similarity/AM-II-filtered_with_labels_k4_test_simi

[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerator
[23:22:13] DEPRECATION WARNING: please use MorganGenerat

阈值 0.90: 样本数=46, 占比=6.87%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim90.csv
阈值 0.80: 样本数=65, 占比=9.70%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim80.csv
阈值 0.70: 样本数=207, 占比=30.90%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim70.csv
阈值 0.60: 样本数=476, 占比=71.04%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim60.csv
阈值 0.50: 样本数=629, 占比=93.88%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim50.csv
阈值 0.40: 样本数=667, 占比=99.55%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim40.csv
阈值 0.30: 样本数=670, 占比=100.00%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim30.csv
阈值 0.20: 样本数=670, 占比=100.00%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim20.csv
阈值 0.10: 样本数=670, 占比=100.00%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim10.csv
阈值 0.00: 样本数=670, 占比=100.00%, 已保存 ./3-similarity/AM-I-filtered_with_labels_k4_test_sim0_all.csv
📑 已保存统计结果和分位数: ./3-similarity/AM-I-filtered_with_labels_k4_test_similarity_st

# 全部区间统计

In [5]:
import os
import glob
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from matplotlib.colors import LinearSegmentedColormap

# ================= Configuration =================
DATA_FOLDER     = './3-similarity'
OUTPUT_FOLDER   = './3-similarity-prediction'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

MODEL_FOLDERS = {
    'AM-I-filtered_with_labels_k4': './2-svr-models/AM-I-svr-model',
    'AM-II-filtered_with_labels_k4': './2-svr-models/AM-II-svr-model'
}
SUMMARY_CSV_PATH = os.path.join(OUTPUT_FOLDER, 'similarity_prediction_summary.csv')
BOXPLOT_PATH    = os.path.join(OUTPUT_FOLDER, 'mae_mre_boxplot.png')
BOXSUMMARY_PATH = os.path.join(OUTPUT_FOLDER, 'mae_mre_box_summary.csv')

# Feature column definitions (consistent with training script)
NUMERIC_FEATS   = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
MORGAN_FP       = [f'fp_{i}' for i in range(1024)]
OTHER_FP        = [f'col{i}' for i in range(823)]
FEATURE_COLS    = NUMERIC_FEATS + OTHER_FP + MORGAN_FP
TARGET_COL      = 'UV_RT-s'

# ================= 修改点1: 调整相似度阈值范围 =================
# 从原来的10个阈值减少到7个（0.3到0.9）
SIMILARITY_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Collect results
all_eval_results = []
all_pred_details = []

# ---------------------------- Utilities -----------------------------
def get_model_name(file_name: str):
    """
    从文件名提取模型名称
    
    方法学思考：
    1. 使用前缀匹配确定模型类型
    2. 这种方法假设文件名有特定格式，适合有规律的数据集命名
    """
    if file_name.startswith('AM-I-filtered_with_labels_k4_test_sim'):
        return 'AM-I-filtered_with_labels_k4'
    elif file_name.startswith('AM-II-filtered_with_labels_k4_test_sim'):
        return 'AM-II-filtered_with_labels_k4'
    else:
        return None

def extract_similarity(file_name: str):
    """
    从文件名提取相似度值
    
    方法学思考：
    1. 使用正则表达式提取sim后的数字
    2. 除以100转换为小数（假设文件名格式为sim30表示0.30）
    3. 这是一种稳健的字符串解析方法
    """
    m = re.search(r'sim(\d+)', file_name)
    return int(m.group(1)) / 100.0 if m else None

def parse_dataset_info(file_name: str):
    """
    统一解析数据集的模型和相似度信息
    
    方法学思考：
    1. 整合模型和相似度提取逻辑
    2. 提高代码复用性和可维护性
    3. 返回结构化信息便于后续使用
    """
    model_name = get_model_name(file_name)
    similarity = extract_similarity(file_name)
    return model_name, similarity

# ---------------------------- Evaluation Function -----------------------------
def evaluate_svr(model, scaler, X, y_true):
    """
    使用SVR模型进行预测并计算评估指标
    
    方法学思考：
    1. 先对数值特征进行标准化（与训练时一致）
    2. 计算绝对误差和相对误差
    3. 返回详细的统计量和原始误差值用于后续分析
    """
    X_scaled = X.copy()
    X_scaled[:, :len(NUMERIC_FEATS)] = scaler.transform(
        X_scaled[:, :len(NUMERIC_FEATS)])
    y_pred = model.predict(X_scaled)

    abs_err = np.abs(y_true - y_pred)
    rel_err = np.abs((y_true - y_pred) / y_true) * 100

    mae  = abs_err.mean()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mre  = rel_err.mean()

    return y_pred, mae, rmse, mre, abs_err, rel_err

# ---------------------------- Plotting Configuration -----------------------------
# Apple 2023 Color Palette (RGB 8-bit → 0-1)
APPLE_BLUE  = [(0, 102/255, 204/255),  # Dark blue
               (51/255, 153/255, 255/255),
               (102/255, 178/255, 255/255),
               (153/255, 204/255, 255/255),
               (204/255, 229/255, 255/255)]

APPLE_GREEN = [(0, 133/255, 50/255),  # Dark green
               (0, 166/255, 81/255),
               (77/255, 196/255, 110/255),
               (153/255, 221/255, 176/255),
               (230/255, 242/255, 230/255)]

def create_7color_palette(base_palette):
    """
    从基础5色调色板创建7色调色板
    
    方法学思考：
    1. 使用线性插值扩展颜色
    2. 确保颜色平滑过渡
    3. 专为7个相似度阈值设计
    
    颜色分配逻辑：
    - 相似度0.9 → 颜色0（最深）
    - 相似度0.3 → 颜色6（最浅）
    - 线性过渡
    """
    if len(base_palette) >= 7:
        return base_palette[:7]
    
    # 将5色扩展为7色
    # 在关键位置采样：索引位置为0, 1/6, 2/6, 3/6, 4/6, 5/6, 1
    positions = [0, 1/6, 2/6, 3/6, 4/6, 5/6, 1]
    extended = []
    
    for pos in positions:
        # 在基础调色板中的位置
        palette_pos = pos * (len(base_palette) - 1)
        idx = int(palette_pos)
        frac = palette_pos - idx
        
        if idx == len(base_palette) - 1:
            extended.append(base_palette[-1])
        else:
            # 线性插值
            c1 = np.array(base_palette[idx])
            c2 = np.array(base_palette[idx + 1])
            new_color = c1 * (1 - frac) + c2 * frac
            extended.append(tuple(new_color))
    
    return extended[::-1]  # 反转，使0.9对应最深颜色

# ================= 修改点2: 创建7色的调色板（仅AM-I和AM-II） =================
# 每个模型对应7个颜色，对应7个相似度阈值
APPLE_PALETTES = {
    'AM-I-filtered_with_labels_k4': create_7color_palette(APPLE_BLUE),
    'AM-II-filtered_with_labels_k4': create_7color_palette(APPLE_GREEN)
}

def get_color_for_dataset(dataset_name: str):
    """
    根据数据集名称获取对应的颜色
    
    方法学思考：
    1. 统一处理模型和相似度信息
    2. 使用查找表方法将相似度映射到颜色索引
    3. 更稳健的颜色分配逻辑
    
    修改点3: 调整颜色索引计算方式
    - 使用相似度值直接计算索引
    - 相似度范围: 0.3-0.9 对应索引 0-6
    """
    model_name, similarity = parse_dataset_info(dataset_name)
    
    if model_name is None or similarity is None:
        return 'gray'
    
    if model_name not in APPLE_PALETTES:
        return 'gray'
    
    # 计算颜色索引
    # 相似度: 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9
    # 索引:   6,   5,   4,   3,   2,   1,   0  (因为调色板已反转)
    # 使用线性映射: 相似度从0.3到0.9映射到索引6到0
    
    # 确保相似度在阈值范围内
    if similarity < SIMILARITY_THRESHOLDS[0] or similarity > SIMILARITY_THRESHOLDS[-1]:
        # 如果不在范围内，返回中性颜色
        return 'gray'
    
    # 更精确的索引计算
    # 将相似度映射到0-6的范围
    similarity_normalized = (similarity - SIMILARITY_THRESHOLDS[0]) / (SIMILARITY_THRESHOLDS[-1] - SIMILARITY_THRESHOLDS[0])
    index = int(round(similarity_normalized * (len(SIMILARITY_THRESHOLDS) - 1)))
    
    # 调整索引方向（相似度越大，颜色越深）
    index = len(SIMILARITY_THRESHOLDS) - 1 - index
    
    # 确保索引在有效范围内
    index = max(0, min(len(APPLE_PALETTES[model_name]) - 1, index))
    
    return APPLE_PALETTES[model_name][index]

def generate_fixed_order():
    """
    生成固定的绘图顺序
    
    方法学思考：
    1. 按模型分组，每个模型内按相似度从小到大排列
    2. 这种顺序便于比较不同模型的相同相似度水平
    3. 符合数据可视化最佳实践
    """
    fixed_order = []
    
    # 定义模型顺序
    model_order = ['AM-I-filtered_with_labels_k4', 'AM-II-filtered_with_labels_k4']
    
    for model in model_order:
        # 对每个数据集，按相似度从小到大排列
        for sim in sorted(SIMILARITY_THRESHOLDS):
            sim_str = f"sim{int(sim*100):02d}"
            dataset_name = f"{model}_test_{sim_str}"
            fixed_order.append(dataset_name)
    
    return fixed_order

def plot_boxplots(df_details, save_path):
    """
    绘制箱线图
    
    方法学思考：
    1. 使用固定顺序确保结果一致性
    2. 分开展示绝对误差和相对误差
    3. 添加网格线和分隔线增强可读性
    4. 使用统一的配色方案
    """
    # 生成固定顺序
    FIXED_ORDER = generate_fixed_order()
    
    # 过滤出实际存在的数据集
    existing_datasets = df_details['Dataset'].unique()
    actual_order = [ds for ds in FIXED_ORDER if ds in existing_datasets]
    
    if not actual_order:
        print("⚠️ No datasets found for box plots!")
        return
    
    metrics = ["AbsError", "RelError"]
    titles  = ["MAE (s)", "MRE (%)"]
    
    for metric, title in zip(metrics, titles):
        plt.figure(figsize=(12, 4.8))  # 调整图像宽度（原来14，现在12）
        plt.rcParams['font.size'] = 12
        
        # 准备按顺序排列的数据
        data_sorted, labels_sorted = [], []
        for ds in actual_order:
            if ds in df_details['Dataset'].values:
                data_sorted.append(df_details[df_details['Dataset'] == ds][metric].values)
                labels_sorted.append(ds)
        
        # 创建箱线图
        bp = plt.boxplot(
            data_sorted,
            labels=labels_sorted,
            patch_artist=True,
            flierprops=dict(marker='o', color='red', markersize=6, alpha=0),
            medianprops=dict(linewidth=2.2, color='black'),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5)
        )
        
        # 应用Apple配色方案
        for patch, label in zip(bp['boxes'], labels_sorted):
            patch.set_facecolor(get_color_for_dataset(label))
            patch.set_alpha(0.9)
        
        # 美化图形
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)
        ax.tick_params(axis='both', which='major', width=1.5, length=6)
        ax.tick_params(axis='both', which='minor', width=1.0, length=4)
        
        # 设置y轴范围
        if title.startswith("MAE"):
            plt.ylim(-1, 11)
            ax.set_yticks(range(0, 13, 2))
        else:
            plt.ylim(-1, 13)
            ax.set_yticks(range(0, 13, 2))
        
        plt.ylabel(title, fontsize=14, fontweight='bold')
        
        # 旋转x轴标签并添加分隔线
        plt.xticks(rotation=45, ha='right', fontsize=11)
        plt.yticks(fontsize=11)
        
        # 在不同模型之间添加分隔线
        ax_positions = range(1, len(actual_order) + 1)
        for i in range(len(actual_order)):
            if i < len(actual_order) - 1:
                current_prefix = "_".join(actual_order[i].split("_")[:4])  # AM-I和AM-II的前缀长度一致
                next_prefix = "_".join(actual_order[i+1].split("_")[:4])
                if current_prefix != next_prefix:
                    ax.axvline(x=i + 1.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        
        # 添加网格
        ax.yaxis.grid(True, linestyle='--', alpha=0.7, linewidth=0.5)
        ax.set_axisbelow(True)
        
        plt.tight_layout()
        
        out_path = save_path.replace(".png", f"_{metric}.png")
        plt.savefig(out_path, dpi=600, bbox_inches='tight')
        plt.close()
        print(f"📊 Apple color box plot saved to: {out_path}")
    
    return actual_order

def export_box_summary(df_details, save_path, fixed_order):
    """
    导出箱线图统计信息
    
    方法学思考：
    1. 计算四分位数和IQR等描述性统计量
    2. 按固定顺序组织数据
    3. 便于后续分析和比较
    """
    rows = []
    for metric in ["AbsError", "RelError"]:
        for dataset in fixed_order:
            if dataset not in df_details['Dataset'].values:
                continue
            grp = df_details[df_details['Dataset'] == dataset]
            if len(grp) > 0:
                q1, q2, q3 = np.percentile(grp[metric], [25, 50, 75])
                rows.append({
                    "Dataset": dataset,
                    "Metric": metric,
                    "Q1": q1,
                    "Median": q2,
                    "Q3": q3,
                    "IQR": q3 - q1,
                    "Count": len(grp)
                })
    pd.DataFrame(rows).to_csv(save_path, index=False)
    print(f"📑 Box plot quartile statistics saved to: {save_path}")

def calculate_outlier_percentage(df_details, metric, fixed_order):
    """
    计算异常值百分比
    
    方法学思考：
    1. 使用Tukey方法定义异常值（Q1-1.5*IQR, Q3+1.5*IQR）
    2. 提供数据质量评估指标
    3. 帮助识别模型在不同相似度下的稳健性
    """
    outlier_percentage = {}
    for dataset in fixed_order:
        if dataset not in df_details['Dataset'].values:
            continue
        grp = df_details[df_details['Dataset'] == dataset]
        if len(grp) > 0:
            q1, q3 = np.percentile(grp[metric], [25, 75])
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            outliers = grp[(grp[metric] < lower) | (grp[metric] > upper)]
            outlier_percentage[dataset] = len(outliers) / len(grp) * 100
    return outlier_percentage

def print_similarity_thresholds_info():
    """
    打印相似度阈值信息
    
    方法学思考：
    1. 在程序开始时显示配置信息
    2. 帮助用户确认阈值设置
    3. 提高代码透明度
    """
    print("=" * 80)
    print("相似度阈值配置信息:")
    print(f"阈值数量: {len(SIMILARITY_THRESHOLDS)}")
    print(f"阈值范围: {SIMILARITY_THRESHOLDS[0]} - {SIMILARITY_THRESHOLDS[-1]}")
    print(f"具体阈值: {SIMILARITY_THRESHOLDS}")
    print("=" * 80)
    print()

# ======================== Main Pipeline =========================
def main():
    """
    主函数：执行完整的预测和评估流程
    
    方法学思考：
    1. 模块化处理每个数据集
    2. 统一的错误处理机制
    3. 清晰的进度和结果展示
    4. 结构化的数据保存
    """
    print_similarity_thresholds_info()
    
    csv_files = glob.glob(os.path.join(DATA_FOLDER, '*.csv'))
    
    for csv_file in csv_files:
        base_name = os.path.splitext(os.path.basename(csv_file))[0]
        model_name, similarity = parse_dataset_info(base_name)
        
        if model_name is None or similarity is None:
            continue
        
        # ================= 修改点4: 检查相似度是否在阈值列表中 =================
        if similarity not in SIMILARITY_THRESHOLDS:
            print(f"⚠️ 跳过: {base_name} (相似度 {similarity} 不在阈值列表中)")
            continue
        
        if model_name in ['AM-I-filtered_with_labels_k4', 'AM-II-filtered_with_labels_k4']:
            # SVR模型预测
            model_folder = MODEL_FOLDERS[model_name]
            model_path = os.path.join(model_folder, f"{model_name}_svr_model.joblib")
            scaler_path = os.path.join(model_folder, f"{model_name}_scaler.joblib")
            
            if not (os.path.exists(model_path) and os.path.exists(scaler_path)):
                print(f"⚠️ 警告: 未找到模型文件 {model_name}")
                continue
            
            model = joblib.load(model_path)
            scaler = joblib.load(scaler_path)
            
            df = pd.read_csv(csv_file).dropna(subset=FEATURE_COLS + [TARGET_COL])
            X, y_true = df[FEATURE_COLS].values, df[TARGET_COL].values
            y_pred, mae, rmse, mre, abs_err, rel_err = evaluate_svr(model, scaler, X, y_true)
        else:
            continue
        
        # 保存预测结果
        df['y_pred'] = y_pred
        output_file = os.path.join(OUTPUT_FOLDER, f"{base_name}_predicted.csv")
        df.to_csv(output_file, index=False)
        
        # 收集评估结果
        all_eval_results.append({
            'Dataset': base_name,
            'Model': model_name,
            'Similarity': similarity,
            'MAE': mae,
            'RMSE': rmse,
            'MRE': mre,
            'Sample_Size': len(df)
        })
        
        # 收集详细的误差数据用于箱线图
        tmp = pd.DataFrame({
            "Dataset": base_name,
            "AbsError": abs_err,
            "RelError": rel_err
        })
        all_pred_details.append(tmp)
        
        print(f"{base_name} | {model_name} | sim={similarity:.2f} | "
              f"MAE={mae:.3g} | RMSE={rmse:.3g} | MRE={mre:.3g}% | Samples={len(df)}")
    
    # 保存评估汇总
    if all_eval_results:
        summary_df = pd.DataFrame(all_eval_results)
        
        # 按模型和相似度排序
        summary_df['Sim_Order'] = summary_df['Similarity'].apply(
            lambda x: SIMILARITY_THRESHOLDS.index(x) if x in SIMILARITY_THRESHOLDS else 999
        )
        summary_df['Model_Order'] = summary_df['Model'].apply(
            lambda x: 1 if x == 'AM-I-filtered_with_labels_k4' else 2
        )
        summary_df = summary_df.sort_values(['Model_Order', 'Sim_Order'])
        summary_df = summary_df.drop(['Sim_Order', 'Model_Order'], axis=1)
        
        summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
        print(f"\n✅ 汇总表格已保存到: {SUMMARY_CSV_PATH}")
        
        # 显示汇总统计信息
        print("\n📊 汇总统计信息:")
        print("=" * 80)
        for model in ['AM-I-filtered_with_labels_k4', 'AM-II-filtered_with_labels_k4']:
            model_df = summary_df[summary_df['Model'] == model]
            if not model_df.empty:
                print(f"\n{model}:")
                for sim in SIMILARITY_THRESHOLDS:
                    sim_df = model_df[model_df['Similarity'] == sim]
                    if not sim_df.empty:
                        row = sim_df.iloc[0]
                        print(f"  sim={sim:.1f}: MAE={row['MAE']:.3g}, RMSE={row['RMSE']:.3g}, "
                              f"MRE={row['MRE']:.3g}%, Samples={row['Sample_Size']}")
                    else:
                        print(f"  sim={sim:.1f}: 无数据")
    else:
        print("⚠️ 未生成评估结果!")
        return
    
    # 生成箱线图和统计信息
    if all_pred_details:
        details_df = pd.concat(all_pred_details, ignore_index=True)
        
        # 生成固定顺序
        FIXED_ORDER = generate_fixed_order()
        
        # 创建箱线图
        print(f"\n📈 生成箱线图...")
        actual_order = plot_boxplots(details_df, BOXPLOT_PATH)
        
        if actual_order:
            # 导出箱线图统计信息
            export_box_summary(details_df, BOXSUMMARY_PATH, actual_order)
            
            # 计算并显示异常值百分比
            print("\n📊 异常值分析:")
            print("=" * 80)
            for metric in ["AbsError", "RelError"]:
                print(f"\n{metric} 异常值百分比:")
                outlier_percentage = calculate_outlier_percentage(details_df, metric, actual_order)
                for dataset in actual_order:
                    if dataset in outlier_percentage:
                        # 提取模型和相似度信息以便更好地显示
                        model = "_".join(dataset.split("_")[:4])
                        sim = dataset.split("_")[-1]
                        print(f"  {model} ({sim}): {outlier_percentage[dataset]:.2f}%")
    else:
        print("⚠️ 未收集到用于箱线图的预测详细信息!")

if __name__ == "__main__":
    main()

相似度阈值配置信息:
阈值数量: 7
阈值范围: 0.3 - 0.9
具体阈值: [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

AM-I-filtered_with_labels_k4_test_sim60 | AM-I-filtered_with_labels_k4 | sim=0.60 | MAE=2.58 | RMSE=3.61 | MRE=3.81% | Samples=476
AM-II-filtered_with_labels_k4_test_sim30 | AM-II-filtered_with_labels_k4 | sim=0.30 | MAE=1.92 | RMSE=2.7 | MRE=3.44% | Samples=186
AM-I-filtered_with_labels_k4_test_sim40 | AM-I-filtered_with_labels_k4 | sim=0.40 | MAE=2.95 | RMSE=4.1 | MRE=4.43% | Samples=667
AM-II-filtered_with_labels_k4_test_sim90 | AM-II-filtered_with_labels_k4 | sim=0.90 | MAE=1.81 | RMSE=3.11 | MRE=3.2% | Samples=3
⚠️ 跳过: AM-I-filtered_with_labels_k4_test_sim20 (相似度 0.2 不在阈值列表中)
⚠️ 跳过: AM-I-filtered_with_labels_k4_test_sim0_all (相似度 0.0 不在阈值列表中)
AM-II-filtered_with_labels_k4_test_sim40 | AM-II-filtered_with_labels_k4 | sim=0.40 | MAE=1.89 | RMSE=2.67 | MRE=3.39% | Samples=183
AM-II-filtered_with_labels_k4_test_sim80 | AM-II-filtered_with_labels_k4 | sim=0.80 | MAE=0.877 | RMSE=1.67 | MRE=1.66% | Samples=12


/tmp/ipykernel_260515/3379997921.py:268: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = plt.boxplot(


📊 Apple color box plot saved to: ./3-similarity-prediction/mae_mre_boxplot_AbsError.png


/tmp/ipykernel_260515/3379997921.py:268: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = plt.boxplot(


📊 Apple color box plot saved to: ./3-similarity-prediction/mae_mre_boxplot_RelError.png
📑 Box plot quartile statistics saved to: ./3-similarity-prediction/mae_mre_box_summary.csv

📊 异常值分析:

AbsError 异常值百分比:
  AM-I-filtered_with_labels_k4 (sim30): 6.42%
  AM-I-filtered_with_labels_k4 (sim40): 6.60%
  AM-I-filtered_with_labels_k4 (sim50): 6.20%
  AM-I-filtered_with_labels_k4 (sim60): 5.04%
  AM-I-filtered_with_labels_k4 (sim70): 4.35%
  AM-I-filtered_with_labels_k4 (sim80): 1.54%
  AM-I-filtered_with_labels_k4 (sim90): 4.35%
  AM-II-filtered_with_labels_k4 (sim30): 6.99%
  AM-II-filtered_with_labels_k4 (sim40): 7.65%
  AM-II-filtered_with_labels_k4 (sim50): 7.14%
  AM-II-filtered_with_labels_k4 (sim60): 6.19%
  AM-II-filtered_with_labels_k4 (sim70): 4.00%
  AM-II-filtered_with_labels_k4 (sim80): 8.33%
  AM-II-filtered_with_labels_k4 (sim90): 0.00%

RelError 异常值百分比:
  AM-I-filtered_with_labels_k4 (sim30): 7.61%
  AM-I-filtered_with_labels_k4 (sim40): 7.50%
  AM-I-filtered_with_labels_k4 (